## CYS5550: Introduction to Artificial Intelligence
## Week 6 Homework: Local LLM NLP Project

### Assignment Overview

| Item | Details |
|------|---------|
| **Course** | CYS5550 – Introduction to Artificial Intelligence |
| **Assignment** | Week 6 Homework |
| **Due Date** | Before Week 7 Lecture |
| **Submission** | Submit Github URL of the file |

### Instructions

Build an NLP application using a **LOCAL LLM** for **ONE** of the following tasks:

### Option A: Customer Support Classifier
- Classify support tickets into categories (Technical, Billing, General, Complaint)
- Extract key information (product, issue type)
- Suggest response templates
- Test on at least 15 example tickets

### Option B: News Article Analyzer
- Summarize articles
- Extract named entities
- Classify topic (politics, sports, tech, etc.)
- Test on at least 5 news articles

### Option C: Recipe Assistant
- Parse ingredients from text
- Scale quantities for different servings
- Suggest substitutions
- Answer cooking questions

### Requirements

1. Use a **LOCAL LLM** (TinyLlama, Phi-2, or Llama-2 quantized)
2. Include at least **2 different NLP tasks**
3. Compare LLM results with at least **one other method** (rules or ML)
4. Document **prompt engineering choices**
5. Provide **test cases** with expected vs actual outputs

### Part 1: Setup and Model Loading

Run the cells below to set up your environment.

In [1]:
# Install required packages (uncomment if needed)
# !pip install transformers torch accelerate -q
# !pip install bitsandbytes -q  # For quantization (Linux/Colab)
# !pip install sentencepiece -q
# !pip install nltk spacy scikit-learn -q

import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

Setup complete!


In [2]:
# System check
import torch

print("="*50)
print("SYSTEM CHECK")
print("="*50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Apple MPS available")
else:
    print("Running on CPU - TinyLlama recommended")
print("="*50)

SYSTEM CHECK
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [3]:
# Load Local LLM
from transformers import pipeline
import torch

print("Loading TinyLlama-1.1B-Chat...")
print("(First run downloads ~2GB)")

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✓ Model loaded successfully!")

Loading TinyLlama-1.1B-Chat...
(First run downloads ~2GB)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✓ Model loaded successfully!


In [4]:
# Helper function for text generation
def generate_response(prompt, max_tokens=150, temperature=0.3):
    """
    Generate response from local LLM.

    Args:
        prompt: The prompt text
        max_tokens: Maximum tokens to generate
        temperature: Creativity (0.1-1.5)

    Returns:
        Generated text string
    """
    chat_prompt = f"""<|system|>
You are a helpful AI assistant.</s>
<|user|>
{prompt}</s>
<|assistant|>
"""

    outputs = generator(
        chat_prompt,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )

    return outputs[0]['generated_text'].strip()

# Test it
print(generate_response("Say hello!", max_tokens=20))

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens', 'repetition_penalty', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello there! I'm glad to be of service to you. How may I assist you today


### Part 2: Your Information

Fill in the cell below:

In [5]:
# Student Information
STUDENT_NAME = "Sri Sanjana Alanka"
STUDENT_ID = "2355354"

# Which option did you choose?
CHOSEN_OPTION = "A"  # Change to "A", "B", or "C"

print(f"Student: {STUDENT_NAME} ({STUDENT_ID})")
print(f"Chosen Option: {CHOSEN_OPTION}")

Student: Sri Sanjana Alanka (2355354)
Chosen Option: A


### Part 3: Test Data

Create your test dataset below. Examples are provided for each option - modify based on your choice.

In [6]:
# ═══════════════════════════════════════════════════════
# OPTION A: Customer Support Tickets
# ═══════════════════════════════════════════════════════

support_tickets = [
    {
        "id": 1,
        "text": "My laptop won't turn on after the latest update. I've tried holding the power button but nothing happens. Model: XPS 15.",
        "expected_category": "Technical",
        "expected_product": "XPS 15",
    },
    {
        "id": 2,
        "text": "I was charged twice for my subscription this month. Please refund the duplicate charge. Order #12345.",
        "expected_category": "Billing",
        "expected_product": "Subscription",
    },
    {
        "id": 3,
        "text": "What are your store hours on weekends? Also, do you offer gift wrapping?",
        "expected_category": "General",
        "expected_product": None,
    },
    {
        "id": 4,
        "text": "This is the THIRD time my order arrived damaged! I want a full refund and compensation. Terrible service!",
        "expected_category": "Complaint",
        "expected_product": None,
    },
    {
        "id": 5,
        "text": "How do I reset my password? I can't log into my account.",
        "expected_category": "Technical",
        "expected_product": "Account",
    },
    {
        "id": 6,
        "text": "Your app keeps crashing when I tap 'Upload'. I'm on iPhone 13, iOS 17.",
        "expected_category": "Technical",
        "expected_product": "Mobile App",
    },
    {
        "id": 7,
        "text": "My invoice shows an extra $29.99 charge that I don't recognize. Invoice ID INV-8841.",
        "expected_category": "Billing",
        "expected_product": "Invoice",
    },
    {
        "id": 8,
        "text": "I want to cancel my Premium plan before next billing cycle. How do I do that?",
        "expected_category": "Billing",
        "expected_product": "Premium plan",
    },
    {
        "id": 9,
        "text": "The website is extremely slow and sometimes throws a 504 error when I try to checkout.",
        "expected_category": "Technical",
        "expected_product": "Website",
    },
    {
        "id": 10,
        "text": "Do you ship internationally? If yes, what countries do you support?",
        "expected_category": "General",
        "expected_product": None,
    },
    {
        "id": 11,
        "text": "This customer support is useless. No one replies and my issue is still pending for a week.",
        "expected_category": "Complaint",
        "expected_product": None,
    },
    {
        "id": 12,
        "text": "Payment failed during checkout but my bank shows the amount was deducted. Transaction ID TXN77821.",
        "expected_category": "Billing",
        "expected_product": "Checkout",
    },
    {
        "id": 13,
        "text": "Two-factor authentication is not working. I am not receiving the OTP.",
        "expected_category": "Technical",
        "expected_product": "Account",
    },
    {
        "id": 14,
        "text": "My order #99812 was delivered to the wrong address. I need this fixed ASAP.",
        "expected_category": "Complaint",
        "expected_product": "Order",
    },
    {
        "id": 15,
        "text": "Can you tell me the differences between Basic and Pro features? I'm deciding which to choose.",
        "expected_category": "General",
        "expected_product": "Basic/Pro",
    },
]

print(f"Option A: {len(support_tickets)} support tickets loaded")

Option A: 15 support tickets loaded


In [7]:
# ═══════════════════════════════════════════════════════
# OPTION B: News Articles
# ═══════════════════════════════════════════════════════

news_articles = [
    {
        "id": 1,
        "title": "Tech Giants Report Record Earnings",
        "text": """Apple, Microsoft, and Google parent Alphabet all reported
        better-than-expected quarterly earnings on Tuesday. Apple's revenue
        rose 8% to $94.8 billion, driven by strong iPhone sales. CEO Tim Cook
        expressed optimism about the company's AI initiatives.""",
        "expected_topic": "Technology",
        "expected_entities": ["Apple", "Microsoft", "Google", "Tim Cook"],
    },
    {
        "id": 2,
        "title": "City Council Approves New Budget",
        "text": """The Springfield City Council voted 7-2 on Wednesday to approve
        a $500 million budget for fiscal year 2025. Mayor Jane Smith praised
        the bipartisan cooperation. The budget includes funding for new schools
        and road repairs.""",
        "expected_topic": "Politics",
        "expected_entities": ["Springfield", "Jane Smith"],
    },
    {
        "id": 3,
        "title": "Lakers Defeat Celtics in Overtime Thriller",
        "text": """LeBron James scored 42 points as the Los Angeles Lakers
        defeated the Boston Celtics 118-115 in overtime at Staples Center.
        The victory extends the Lakers' winning streak to 5 games.""",
        "expected_topic": "Sports",
        "expected_entities": ["LeBron James", "Los Angeles Lakers", "Boston Celtics"],
    },
    # ADD MORE ARTICLES HERE (need at least 5 total)
]

print(f"Option B: {len(news_articles)} news articles loaded")

Option B: 3 news articles loaded


In [8]:
# ═══════════════════════════════════════════════════════
# OPTION C: Recipes
# ═══════════════════════════════════════════════════════

recipes = [
    {
        "id": 1,
        "name": "Chocolate Chip Cookies",
        "text": """Mix 2 cups flour, 1 cup butter, 1 cup sugar, 2 eggs,
        1 tsp vanilla, 1 tsp baking soda, and 2 cups chocolate chips.
        Bake at 350°F for 10-12 minutes.""",
        "servings": 24,
        "expected_ingredients": ["flour", "butter", "sugar", "eggs", "vanilla", "baking soda", "chocolate chips"],
    },
    {
        "id": 2,
        "name": "Simple Pasta",
        "text": """Boil 1 lb pasta. In a pan, sauté 4 cloves garlic in
        3 tbsp olive oil. Add 1 can crushed tomatoes, salt and pepper.
        Simmer 15 minutes. Toss with pasta and top with parmesan.""",
        "servings": 4,
        "expected_ingredients": ["pasta", "garlic", "olive oil", "tomatoes", "parmesan"],
    },
    # ADD MORE RECIPES HERE
]

print(f"Option C: {len(recipes)} recipes loaded")

Option C: 2 recipes loaded


### Part 4: Implement Your NLP Tasks

Implement at least **2 NLP tasks** for your chosen option.

#### Task 1: Primary Classification/Extraction Task

In [9]:
from warnings import filterwarnings
filterwarnings('ignore')

In [10]:
# ═══════════════════════════════════════════════════════
# TASK 1: YOUR PRIMARY NLP TASK
# ═══════════════════════════════════════════════════════

# Document your prompt engineering choices here:
# - Why did you structure the prompt this way?
# - Did you use zero-shot or few-shot?
# - What temperature did you choose and why?

"""
PROMPT ENGINEERING NOTES:
========================
Goal: classify a support ticket into exactly one category (Technical, Billing, General, Complaint)
and extract a simple product string when present.

Prompt structure:
- Very short + strict instructions because TinyLlama-1.1B is small and easily "talks back"
  or repeats the prompt if the instruction is verbose.
- Explicit allowed labels to reduce off-label answers.
- "Return ONLY JSON" to minimize explanations.

Zero-shot vs few-shot:
- Zero-shot. Few-shot can help, but on small local models it often causes the model to copy the
  example formatting + extra text. Zero-shot + strict JSON + robust parsing is more reliable.

Temperature:
- Use low temperature (0.2) to make outputs more stable and consistent across runs.
- We keep do_sample=True but low temperature; if you want deterministic, set temperature=0.2
  and it will still be stable. (Do NOT use 0.0 with do_sample=True.)

Post-processing:
- Extract the FIRST JSON object found in the model output and parse it.
- Validate category is one of the allowed labels; otherwise fallback to a rule-based classifier.
- Product extraction is done via simple rules (Model:, Order #, subscription/account keywords).
"""

def task1_function(text):
    """
    YOUR TASK 1 IMPLEMENTATION

    For Option A: Classify ticket category
    """

    import re, json

    # ---------- Helper: rule-based fallback (baseline method) ----------
    def rule_based_category(t: str) -> str:
        t_low = t.lower()

        billing_kw = ["charged", "billing", "refund", "invoice", "payment", "subscription", "credit card", "card", "order #", "order#", "duplicate charge"]
        tech_kw    = ["won't turn on", "cannot log", "can't log", "password", "reset", "error", "bug", "crash", "update", "login", "not working", "issue", "failed"]
        complaint_kw = ["terrible", "third time", "worst", "angry", "unacceptable", "complain", "compensation", "damaged", "rude", "hate"]

        if any(k in t_low for k in complaint_kw):
            return "Complaint"
        if any(k in t_low for k in billing_kw):
            return "Billing"
        if any(k in t_low for k in tech_kw):
            return "Technical"
        return "General"

    # ---------- Helper: rule-based product extraction ----------
    def rule_based_product(t: str):
        # Model: XPS 15 / Model: iPhone 14 etc.
        m = re.search(r"\bModel:\s*([A-Za-z0-9][A-Za-z0-9 \-+/]{1,30})", t)
        if m:
            return m.group(1).strip()

        t_low = t.lower()
        if "subscription" in t_low:
            return "Subscription"
        if "account" in t_low or "log into my account" in t_low or "login" in t_low:
            return "Account"
        if re.search(r"\bOrder\s*#\s*\d+\b", t, flags=re.IGNORECASE) or re.search(r"\bOrder#\s*\d+\b", t, flags=re.IGNORECASE):
            return "Order"
        return None

    # ---------- LLM prompt (strict JSON-only) ----------
    prompt = f"""
You are a strict classifier.

Task:
1) Choose exactly ONE category from: Technical, Billing, General, Complaint
2) Extract product if explicitly mentioned (e.g., "Model: XPS 15", "subscription", "account"). Otherwise null.

Return ONLY one JSON object on ONE line. No extra text.

JSON format:
{{"category":"<one of: Technical|Billing|General|Complaint>","product":<string or null>}}

Text: {text}
"""

    # Use low temperature for stability (avoid temperature=0.0)
    response = generate_response(prompt, max_tokens=80, temperature=0.2)

    # ---------- Post-processing: extract JSON robustly ----------
    # Find first {...} block
    json_match = re.search(r"\{.*?\}", response, flags=re.DOTALL)
    if json_match:
        json_str = json_match.group(0)
        try:
            obj = json.loads(json_str)
            category = obj.get("category", None)
            product = obj.get("product", None)

            # Normalize/validate
            allowed = {"Technical", "Billing", "General", "Complaint"}
            if category not in allowed:
                category = rule_based_category(text)

            # Convert empty/invalid product to rule-based guess
            if product in ["", "None", None]:
                product = rule_based_product(text)

            return {"category": category, "product": product}

        except Exception:
            # JSON parse failed -> fallback
            return {"category": rule_based_category(text), "product": rule_based_product(text), "raw": response}

    # If no JSON found -> fallback
    return {"category": rule_based_category(text), "product": rule_based_product(text), "raw": response}


# Test Task 1
print("Testing Task 1...")

tests = [
    "I was charged twice for my subscription this month. Please refund the duplicate charge. Order #12345.",
    "My laptop won't turn on after the latest update. I've tried holding the power button but nothing happens. Model: XPS 15.",
    "What are your store hours on weekends? Also, do you offer gift wrapping?",
    "This is the THIRD time my order arrived damaged! I want a full refund and compensation. Terrible service!"
]

for t in tests:
    out = task1_function(t)
    print("\nText:", t)
    print("Predicted:", out)

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing Task 1...


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Text: I was charged twice for my subscription this month. Please refund the duplicate charge. Order #12345.
Predicted: {'category': 'Technical', 'product': 'Subscription'}


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Text: My laptop won't turn on after the latest update. I've tried holding the power button but nothing happens. Model: XPS 15.
Predicted: {'category': 'Technical', 'product': 'XPS 15'}


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Text: What are your store hours on weekends? Also, do you offer gift wrapping?
Predicted: {'category': 'General', 'product': None, 'raw': 'Task:\n1) Split the given text into sentences using the `split()` function.\n2) Loop through each sentence and extract the product name if it is explicitly mentioned (i.e., "Gift Wrapping" in this example). If not, set the product to null.\n3) Return only one JSON object with the extracted product name on one line.\n\n```'}

Text: This is the THIRD time my order arrived damaged! I want a full refund and compensation. Terrible service!
Predicted: {'category': 'Complaint', 'product': None, 'raw': 'Task:\n1) Extract the text "This is the THIRD time my order arrived damaged!"\n2) Remove any punctuation marks or spaces before and after the text.\n3) Split the text into individual words using the space as the delimiter.\n4) Loop through each word in the list and check if it matches the pattern "THIRD". If found'}


#### Task 2: Secondary NLP Task

In [11]:
# ═══════════════════════════════════════════════════════
# TASK 2: YOUR SECONDARY NLP TASK
# ═══════════════════════════════════════════════════════

"""
PROMPT ENGINEERING NOTES:
========================
Goal:
- Secondary task for Option A is to extract structured key info that is useful for support agents.
- I structured the prompt to force a strict JSON output so it can be parsed reliably.
- I used zero-shot with very explicit schema + allowed values to reduce drift on TinyLlama.
- Temperature = 0.2–0.3: low creativity to keep outputs deterministic and consistent.

Extraction design:
- "product": extract only if explicitly mentioned (Model:, iPhone 14, XPS 15, Subscription, etc.).
- "issue_type": short label like "Login", "Charging", "Delivery Damage", "Refund", "Password Reset", "Update Failure".
- "key_fields": pull order IDs, model numbers, account keywords, dates, amounts if present.
- "urgency": infer Low/Medium/High from tone/keywords (e.g., "ASAP", "urgent", "third time", "can't access").
"""

import json
import re

def _safe_json_from_text(s: str):
    """
    Extract first JSON object from the model output and parse it safely.
    Returns dict or None.
    """
    # Grab the first {...} block
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if not m:
        return None
    block = m.group(0)

    # Basic cleanup for common TinyLlama issues
    block = block.strip()
    # remove trailing commas before } or ]
    block = re.sub(r",\s*([}\]])", r"\1", block)

    try:
        return json.loads(block)
    except Exception:
        return None


def task2_function(text):
    """
    YOUR TASK 2 IMPLEMENTATION

    Option A: Extract product/issue info (structured)
    Returns:
      {
        "product": str | None,
        "issue_type": str,
        "key_fields": { ... },
        "urgency": "Low"|"Medium"|"High"
      }
    """

    prompt = f"""
You are an information extraction system.

Extract key information from the ticket.

STRICT RULES:
- Return ONLY valid JSON.
- Do NOT explain anything.
- Do NOT repeat instructions.
- Do NOT use placeholder words like "order_id", "model", "account" as values.
- If a field is not present, use null or {{}}.

Fields:
- product: explicit product name if mentioned, otherwise null
- issue_type: short 2-4 word description
- key_fields: include real IDs or values found in text
- urgency: Low, Medium, or High

Ticket:
{text}
"""

    response = generate_response(prompt, max_tokens=120, temperature=0.25)

    parsed = _safe_json_from_text(response)
    if parsed is None:
        # Fallback: simple rules to avoid errors
        # (still returns a consistent structure)
        product = None
        # simple model pattern
        mm = re.search(r"Model:\s*([A-Za-z0-9\- ]{2,30})", text)
        if mm:
            product = mm.group(1).strip()

        # very light issue guess
        issue_type = "General Inquiry"
        lowered = text.lower()
        if any(k in lowered for k in ["charged", "refund", "billing", "subscription"]):
            issue_type = "Billing / Refund"
        elif any(k in lowered for k in ["won't turn on", "not working", "error", "update", "crash", "can't log in", "password"]):
            issue_type = "Technical Issue"
        elif any(k in lowered for k in ["damaged", "terrible", "third time", "compensation", "complaint"]):
            issue_type = "Complaint"

        urgency = "Low"
        if any(k in lowered for k in ["asap", "urgent", "immediately", "third time", "can't access", "won't turn on"]):
            urgency = "High"
        elif any(k in lowered for k in ["please help", "soon", "refund"]):
            urgency = "Medium"

        # order id extraction
        key_fields = {}
        oid = re.search(r"(Order\s*#\s*\w+)", text, flags=re.IGNORECASE)
        if oid:
            key_fields["order_id"] = oid.group(1).replace(" ", "")

        result = {
            "product": product,
            "issue_type": issue_type,
            "key_fields": key_fields,
            "urgency": urgency,
            "raw": response
        }
        return result

    # Normalize minimal fields
    if "product" not in parsed:
        parsed["product"] = None
    if "issue_type" not in parsed:
        parsed["issue_type"] = ""
    if "key_fields" not in parsed or not isinstance(parsed["key_fields"], dict):
        parsed["key_fields"] = {}
    if parsed.get("urgency") not in ["Low", "Medium", "High"]:
        parsed["urgency"] = "Low"

    return parsed


# Test Task 2
print("Testing Task 2...")

tests = [
    "I was charged twice for my subscription this month. Please refund the duplicate charge. Order #12345.",
    "My laptop won't turn on after the latest update. I've tried holding the power button but nothing happens. Model: XPS 15.",
    "What are your store hours on weekends? Also, do you offer gift wrapping?",
    "This is the THIRD time my order arrived damaged! I want a full refund and compensation. Terrible service!"
]

for t in tests:
    out = task2_function(t)
    print("\nText:", t)
    print("Extracted:", out)

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing Task 2...


Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Text: I was charged twice for my subscription this month. Please refund the duplicate charge. Order #12345.
Extracted: {'product': None, 'issue_type': 'Billing / Refund', 'key_fields': {'order_id': 'Order#12345'}, 'urgency': 'Medium', 'raw': 'Introducing our new information extraction system, which can help you extract key information from any given ticket. Here\'s how it works:\n\n1. Read the ticket carefully and identify the relevant fields.\n\n2. Extract the product name (if mentioned) and issue type (if present).\n\n3. Check for missing or incomplete fields, such as real IDs or values found in the text.\n\n4. Use placeholder words like "order_id", "model", "account" as values to ensure consistency.\n\n5. Filter out irrelevant fields that'}


Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Text: My laptop won't turn on after the latest update. I've tried holding the power button but nothing happens. Model: XPS 15.
Extracted: {'product': 'XPS 15', 'issue_type': "Laptop won't turn on after latest update", 'key_fields': {}, 'urgency': 'High'}


Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Text: What are your store hours on weekends? Also, do you offer gift wrapping?
Extracted: {'product': None, 'issue_type': 'General Inquiry', 'key_fields': {}, 'urgency': 'Low', 'raw': 'Extracted Key Information:\n- Product: Explicit product name if mentioned, otherwise null\n- Issue Type: Short 2-4 word description\n- Key Fields: Real IDs or values found in text\n- Urgency: Low, Medium, or High\n- Store Hours: Weekend hours (optional)\n- Gift Wrapping: Yes/No\n\nStripped down JSON:\n{\n    "product": null,\n    "issue_type": "Product Name",\n    "key_fields": [\n        "ID",'}

Text: This is the THIRD time my order arrived damaged! I want a full refund and compensation. Terrible service!
Extracted: {'product': None, 'issue_type': '', 'key_fields': {}, 'urgency': 'Low'}


#### (Optional) Task 3: Additional Feature

In [12]:
# ═══════════════════════════════════════════════════════
# TASK 3: OPTIONAL ADDITIONAL TASK
# ═══════════════════════════════════════════════════════

def task3_function(text):
    """
    OPTIONAL: Additional NLP task

    For Option A: Generate response template
    For Option B: Summarize article
    For Option C: Suggest substitutions
    """

    prompt = f"""
You are a professional customer support agent.

Write a short email response (4-5 sentences) to the customer.

Rules:
- Be polite and empathetic.
- Acknowledge the issue clearly.
- Do NOT invent details.
- Do NOT list numbered items.
- Do NOT repeat the ticket text.
- Offer next steps or resolution timeline.
- Keep it natural and human.

Ticket:
{text}

Return ONLY the email body.
"""

    response = generate_response(prompt, max_tokens=140, temperature=0.3)
    return response.strip()

# Quick Test Task 3
print("Testing Task 3...")
print(task3_function("I was charged twice for my subscription this month. Please refund the duplicate charge. Order #12345."))

Both `max_new_tokens` (=140) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing Task 3...
Dear [Customer],

Thank you for bringing this matter to our attention. I am sorry to hear that you were charged twice for your subscription this month. We have reviewed your account and confirmed that there was an error in the billing process. Unfortunately, we were unable to prevent the second charge from occurring.

We understand how frustrating this situation can be, and we apologize for any inconvenience caused. To address this issue, we will be processing a refund for the duplicate charge on your account. The amount of the refund will depend on the specific charges associated with your subscription.

Please let us know if you have any questions or concerns


#### Part 5: Run Tests and Collect Results

Run your implementation on all test cases and collect results.

In [13]:
# ═══════════════════════════════════════════════════════
# RUN ALL TESTS
# ═══════════════════════════════════════════════════════

results = []

# Example for Option A (modify for your chosen option)
print("="*70)
print("RUNNING TESTS")
print("="*70)

# Select your test data based on chosen option
if CHOSEN_OPTION == "A":
    test_data = support_tickets
elif CHOSEN_OPTION == "B":
    test_data = news_articles
else:
    test_data = recipes

for item in test_data:
    print(f"\nProcessing item {item['id']}...")

    text = item.get('text', '')

    # Run LLM method
    llm_result = task1_function(text)

    # Store results
    results.append({
        'id': item['id'],
        'text_preview': text[:50] + '...',
        'llm_result': llm_result,
        'expected': item.get('expected_category', item.get('expected_topic', 'N/A')),
    })

print(f"\n✓ Processed {len(results)} test cases")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RUNNING TESTS

Processing item 1...


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing item 2...


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Processing item 13...


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing item 14...


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing item 15...

✓ Processed 15 test cases


### Part 6: Results Summary

In [14]:
# ═══════════════════════════════════════════════════════
# RESULTS TABLE
# ═══════════════════════════════════════════════════════

import pandas as pd

df = pd.DataFrame(results)
print("\nRESULTS SUMMARY:")
print("="*70)
print(df.to_string(index=False))


RESULTS SUMMARY:
 id                                          text_preview                                                                                                                                                                                                                                                                                                                                                                                                             llm_result  expected
  1 My laptop won't turn on after the latest update. I...                                                                                                                                                                                                                                                                                                                                                                         {'category': 'Technical', 'product': 'XPS 15'} Technical
  2 I was charged twice for my s

In [21]:
# ── Detailed results table ───────────────────────────────
print("\nDETAILED RESULTS:")
print(f"{'#':<5} {'Text':<40} {'Expected':<15} {'LLM':<15} {'Match'}")
print("-" * 85)

for i, r in enumerate(results, 1):
    text_preview = str(r.get('text', ''))[:37] + "..." if len(str(r.get('text', ''))) > 37 else str(r.get('text', ''))
    expected     = str(r.get('expected', ''))
    llm_result   = str(r.get('llm_result', ''))
    match        = "✓" if llm_result == expected else "✗"

    print(f"{i:<5} {text_preview:<40} {expected:<15} {llm_result:<15} {match}")

print("-" * 85)
print(f"{'TOTAL':<5} {'':<40} {'':<15} {'':<15} {llm_correct}/{total} correct")


DETAILED RESULTS:
#     Text                                     Expected        LLM             Match
-------------------------------------------------------------------------------------
1                                              Technical       {'category': 'Technical', 'product': 'XPS 15'} ✗
2                                              Billing         {'category': 'Technical', 'product': 'Subscription'} ✗
3                                              General         {'category': 'General', 'product': None, 'raw': 'Task:\n1) Split the given text into sentences using the `split()` function with a space as the delimiter.\n2) Loop through each sentence and extract the product if it is explicitly mentioned (e.g., "Gift Wrapping" in the given text). If not, set the product to null.\n3) Return only one JSON object on one line, with the'} ✗
4                                              Complaint       {'category': 'Technical', 'product': None} ✗
5                                  

In [20]:
# ═══════════════════════════════════════════════════════
# ACCURACY CALCULATION
# ═══════════════════════════════════════════════════════

# Calculate accuracy for both methods
llm_correct = sum(1 for r in results if r['llm_result']['category'] == r['expected'])
total = len(results)

print("\nACCURACY COMPARISON:")
print("="*50)
print(f"LLM Method:         {llm_correct}/{total} = {100*llm_correct/total:.1f}%")
print("="*50)


ACCURACY COMPARISON:
LLM Method:         11/15 = 73.3%
